# 01. Azure Machine Learning 環境構築

**対応するテキスト**: [docs/03_AzureML環境構築.md](../docs/03_AzureML環境構築.md)

このノートブックで行うこと:

1. ワークスペースへの接続（**Azure へのサインインもこの中で行います**）
2. コンピューティング クラスターの作成（**`min_instances=0`**）
3. カスタム環境の作成（[../src/conda.yaml](../src/conda.yaml)）
4. MLflow 追跡 URI の設定
5. **疎通確認ジョブ**（ロボット環境が Linux 上で動くかの確認を含む）

> [!WARNING]
> **このノートブックは Azure 上で実行検証していません。**
> 本ハンズオンの構築時に検証したのは**ローカル（Windows）実行だけ**です。
> そのため、Azure ジョブの所要時間・費用・出力例は**一切記載していません**。

> [!IMPORTANT]
> **⚠ 最大のリスクは「PyBullet が Linux のコンテナーで動くか」です。**
> ローカル（Windows）では conda-forge の `pybullet` を使いますが、
> Azure ML では **PyPI の manylinux ホイール**を使います（[docs/03 の 3.4](../docs/03_AzureML環境構築.md)）。
> **この組み合わせは未検証です。** 6. の疎通確認ジョブで最初に確かめてください。

> [!IMPORTANT]
> **先に [docs/02_環境を準備する.md](../docs/02_環境を準備する.md) を完了してください。**
> このノートブックは conda 環境 `il-panda` のカーネル（**Python (il-panda)**）で実行します。

> ⚠ **書き換えたノートブックをそのままコミットしないでください。**

## 1. ワークスペース情報の入力

**取得方法**: [Azure ML studio](https://ml.azure.com) の右上にあるワークスペース名をクリックすると、
サブスクリプション ID・リソース グループ・ワークスペース名が表示されます。

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

#  本ハンズオンで作成・利用するリソースの名前（以降のノートブックでも同じ名前を使います）
COMPUTE_NAME = "cpu-cluster"
COMPUTE_SIZE = "Standard_DS3_v2"      # CPU 4 コア。クォータに合わせて調整してください
MAX_INSTANCES = 4                     # 並列実行できるジョブ数の上限
ENVIRONMENT_NAME = "il-pickplace-env"

#  コスト集計用のタグ。後片付け（09 章）でフィルターに使います
TAGS = {
    "project": "il-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

print("設定を読み込みました。")

## 2. Azure へのサインインとワークスペースへの接続

**ターミナルで `az login` を実行しなくても構いません。** ここでサインインできます。

| `AUTH_MODE` | 使うクラス | 動作 |
|---|---|---|
| `"auto"`（既定） | `DefaultAzureCredential` | 環境変数・マネージド ID・Azure CLI などを順に試し、どれも使えなければ**ブラウザーを開いて対話サインイン**します |
| `"browser"` | `InteractiveBrowserCredential` | 最初からブラウザーを開きます |
| `"device"` | `DeviceCodeCredential` | **URL と確認コードを表示**します。ブラウザーが使えない環境向け |

> [!IMPORTANT]
> **`"auto"` では `exclude_interactive_browser_credential=False` を明示しています。**
> Python の `DefaultAzureCredential` は、**既定では `InteractiveBrowserCredential` をチェーンから除外する**ためです。
>
> 出典（Microsoft 公式）: [Credential chains in the Azure Identity library for Python](https://learn.microsoft.com/azure/developer/python/sdk/authentication/credential-chains#defaultazurecredential-overview)

In [ ]:
from azure.identity import (
    DefaultAzureCredential,
    DeviceCodeCredential,
    InteractiveBrowserCredential,
)

AUTH_MODE = "auto"   # "auto" | "browser" | "device"

if AUTH_MODE == "auto":
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
elif AUTH_MODE == "browser":
    credential = InteractiveBrowserCredential()
elif AUTH_MODE == "device":
    credential = DeviceCodeCredential()
else:
    raise ValueError(f'AUTH_MODE は "auto" / "browser" / "device" のいずれかです（指定値: {AUTH_MODE}）')

print("認証方式:", type(credential).__name__)

In [ ]:
from azure.ai.ml import MLClient

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

#  ここで初めて Azure へアクセスする。未サインインならこのタイミングで認証が始まる
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("Workspace      :", ws.name)
print("Location       :", ws.location)
print("Resource group :", ws.resource_group)

### ⚠ ここで失敗したら

| 症状 | 対処 |
|---|---|
| トークンを取得できない（`ClientAuthenticationError`） | `AUTH_MODE` を `"browser"` に変えて実行し直す |
| ブラウザーが開かない／リモート セッションで進まない | `AUTH_MODE` を `"device"` に変える |
| サインインは通るがワークスペースが見つからない（`ResourceNotFound`） | 1. の 3 つの名前の綴りを確認する |
| `AuthorizationFailed` | 権限不足。[docs/03_AzureML環境構築.md](../docs/03_AzureML環境構築.md) の 3.2 を参照 |

## 3. コンピューティング クラスターの作成

| パラメーター | 意味 |
|---|---|
| **`min_instances=0`** | **ジョブが無い間はノード数 0 → コンピューティングの課金が止まる** |
| `max_instances` | 並列実行できるジョブ数の上限（**vCPU クォータを超えないこと**） |
| `idle_time_before_scale_down=120` | アイドル 120 秒でノードを解放する |

> 出典（Microsoft 公式）: [Create an Azure Machine Learning compute cluster](https://learn.microsoft.com/azure/machine-learning/how-to-create-attach-compute-cluster?view=azureml-api-2)

> ⚠ **GPU は不要です。** 本ハンズオンの計算は物理シミュレーションが中心で、
> ニューラルネットワークは小さいため、GPU を付けても速くなりません。

In [ ]:
from azure.ai.ml.entities import AmlCompute

try:
    cluster = ml_client.compute.get(COMPUTE_NAME)
    print(f"既存のクラスターを使います: {cluster.name} (size={cluster.size}, max={cluster.max_instances})")
except Exception:
    print("クラスターが無いので作成します...")
    cluster = AmlCompute(
        name=COMPUTE_NAME,
        type="amlcompute",
        size=COMPUTE_SIZE,
        min_instances=0,          # ← 課金を止める最重要の設定
        max_instances=MAX_INSTANCES,
        idle_time_before_scale_down=120,
        tags=TAGS,
    )
    cluster = ml_client.begin_create_or_update(cluster).result()
    print(f"作成しました: {cluster.name}")

print("min_instances:", cluster.min_instances, "（0 であることを確認してください）")

## 4. カスタム環境の作成

[../src/conda.yaml](../src/conda.yaml) をもとに、**ベース Docker イメージ ＋ conda 環境**の形で作成します。

> [!IMPORTANT]
> **Azure ML は conda 定義から新しい環境を作り、その中でジョブを実行します。**
> **ベースイメージに入っている Python パッケージは使えません。**
>
> 出典（Microsoft 公式）: [Manage Azure Machine Learning environments with the CLI and SDK (v2)](https://learn.microsoft.com/azure/machine-learning/how-to-manage-environments-v2?view=azureml-api-2)

> ⚠ **`numpy<2` が必須です**（`panda-gym` の制約）。`conda.yaml` で固定済みです。
> 出典: https://github.com/qgallouedec/panda-gym/blob/master/setup.py

In [ ]:
from azure.ai.ml.entities import Environment

#  Microsoft Learn の環境作成サンプルで使われているベースイメージ
BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env = Environment(
    name=ENVIRONMENT_NAME,
    description="imitation + Stable-Baselines3 + panda-gym + PyBullet + MLflow (pick and place)",
    image=BASE_IMAGE,
    conda_file="../src/conda.yaml",
    tags=TAGS,
)
env = ml_client.environments.create_or_update(env)

ENV_REF = f"{env.name}:{env.version}"
print("作成した環境:", ENV_REF)
print("※ イメージの構築は最初のジョブ投入時に行われます。studio の［環境］→［ビルド ログ］で進捗を確認できます。")

## 5. MLflow 追跡 URI の設定

**Azure ML のコンピューティング上で動くジョブは、追跡先が自動で設定されます。**
一方、**手元の PC から MLflow の記録を読む場合は、明示的に設定が必要**です。

> 出典（Microsoft 公式）: [Azure Machine Learning 用に MLflow を構成する](https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2)
> 「**Azure コンピューティング インフラストラクチャを使用する場合、追跡 URI を構成する必要はありません。**」
> 「**ただし、Azure Machine Learning の外部で作業する場合は、ワークスペースを指すように MLflow を構成する必要があります。**」

In [ ]:
import mlflow

try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    print("[WARN] SDK から追跡 URI を取得できませんでした。ドキュメント記載の形式で組み立てます。")
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )

mlflow.set_tracking_uri(tracking_uri)
print("tracking uri:", tracking_uri[:60], "...")

## 6. 疎通確認ジョブ（**最重要**）

**ここまでの構築が正しいかを、1 本の最小ジョブで確認します。**

確認する項目:

1. `../src` のスクリプトがスナップショットとして送られる
2. **`imitation` / `stable-baselines3` / `panda-gym` / `pybullet` が import できる**
3. ⭐ **Linux 上で PyBullet の物理シミュレーションが動く**
4. **固定ホライズン化した環境が生成できる**
5. **スクリプト専門家がピックアンドプレースに成功する**
6. **MLflow にパラメーターとメトリックが記録される**

> **新しいファイルは作りません。** [../src/il_common.py](../src/il_common.py) と
> [../src/scripted_expert.py](../src/scripted_expert.py) を `python -c` から import するだけで、
> 上の 6 点をすべて確認できます。

In [ ]:
from azure.ai.ml import command

#  疎通確認のためだけに新しいファイルを増やさないよう、python -c の 1 行にまとめます。
SMOKE_CODE = (
    "import importlib.metadata as md;"
    "import mlflow;"
    "from il_common import DEFAULT_ENV_ID, evaluate, make_env;"
    "from scripted_expert import ScriptedExpertPolicy;"
    "venv = make_env(DEFAULT_ENV_ID, 2, 0);"
    "policy = ScriptedExpertPolicy(venv.observation_space, venv.action_space);"
    "mean, std, success = evaluate(policy, venv, 4);"
    "mlflow.log_param('env_id', DEFAULT_ENV_ID);"
    "[mlflow.log_param(n + '_version', md.version(n)) for n in "
    "('imitation', 'stable-baselines3', 'gymnasium', 'panda-gym', 'pybullet', 'numpy', 'torch')];"
    "mlflow.log_metric('obs_dim', float(venv.observation_space.shape[0]));"
    "mlflow.log_metric('action_dim', float(venv.action_space.shape[0]));"
    "mlflow.log_metric('expert_return_mean', mean);"
    "mlflow.log_metric('expert_success_rate', success);"
    "venv.close();"
    "print('SMOKE TEST OK success_rate=', success)"
)

smoke_job = command(
    code="../src",                       # このフォルダー全体がスナップショットとして保存される
    command=f'python -c "{SMOKE_CODE}"',
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    experiment_name="il-setup-check",
    display_name="smoke_test_pick_and_place",
    tags=TAGS,
)

returned_job = ml_client.jobs.create_or_update(smoke_job)
print("ジョブ名 :", returned_job.name)
print("studio  :", returned_job.studio_url)

### 6-1. 完了を待つ

**初回はイメージの構築が入ります。** 上のセルで表示された studio の URL を開くと、進捗とログを確認できます。

In [ ]:
ml_client.jobs.stream(returned_job.name)

job = ml_client.jobs.get(returned_job.name)
print("ステータス:", job.status, "（Completed なら成功）")

### 6-2. MLflow に記録された内容を確認する

ジョブのログに `SMOKE TEST OK` が出ていて、**`expert_success_rate` が 1.0 なら構築は完了**です。

**ローカルでの実測値（参考）**

| メトリック | ローカルでの値 |
|---|---|
| `obs_dim` | 25 |
| `action_dim` | 4 |
| `expert_success_rate` | 1.0 |

> ⚠ `run.data.metrics` は、同じ名前のメトリックについて**最後の値しか返しません。**
> 出典（Microsoft 公式）: [Log metrics, parameters, and files with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2)

In [ ]:
run = mlflow.get_run(returned_job.name)

print("=== パラメーター ===")
for k, v in sorted(run.data.params.items()):
    print(f"  {k:26s}: {v}")

print("\n=== メトリック ===")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k:26s}: {v}")

### ⚠ ここで失敗したら

| 症状 | 対処 |
|---|---|
| 環境（イメージ）のビルドで失敗する | studio の［環境］→［ビルド ログ］を読む。`numpy<2` の解決に失敗していないか確認 |
| `ModuleNotFoundError: pybullet` | `conda.yaml` の `panda-gym` が `pybullet` を引き込めていません |
| **PyBullet の初期化で失敗する** | **Linux コンテナーでの動作は本ハンズオンで未検証です。** ログの全文を確認してください |
| `ResourceNotFound: il-pickplace-env` | 4. のセルを先に実行してください |
| ずっと `Queued` のまま | クォータ不足か、`max_instances` を超える本数を投入しています |
| その他 | [docs/A1_トラブルシューティング.md](../docs/A1_トラブルシューティング.md) |

## 7. ✅ チェックリスト

- [ ] ワークスペースに接続できた
- [ ] **`min_instances=0`** のコンピューティング クラスターがある
- [ ] カスタム環境 `il-pickplace-env` を作成した
- [ ] MLflow 追跡 URI を設定した
- [ ] **疎通確認ジョブが `Completed` で終わり、`SMOKE TEST OK` が出力された**
- [ ] **`expert_success_rate` が 1.0 だった**（= Linux 上で物理シミュレーションが正しく動いている）
- [ ] **サブスクリプション ID を書いたノートブックをコミットしていない**

---

**次へ**: [docs/04_専門家デモを作る.md](../docs/04_専門家デモを作る.md) → [03_collect_demos_job.ipynb](03_collect_demos_job.ipynb)